In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer — функція Qiskit від Kipu Quantum

> **Note:** * Qiskit Functions — це експериментальна функція, доступна лише для користувачів IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього випуску та можуть змінюватися.

## Огляд
За допомогою Iskay Quantum Optimizer від Kipu Quantum ти можеш розв'язувати складні задачі оптимізації на квантових комп'ютерах IBM&reg;. Цей розв'язувач використовує передовий [алгоритм bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) від Kipu, якому потрібна лише цільова функція як вхідні дані для автоматичного отримання розв'язків задач. Він може обробляти задачі оптимізації з кількістю кубітів до 156, що дозволяє використовувати всі кубіти квантових пристроїв IBM. Оптимізатор використовує відображення 1-до-1 між класичними змінними та кубітами, що дозволяє тобі розв'язувати задачі оптимізації з кількістю бінарних змінних до 156.

Оптимізатор дає змогу розв'язувати задачі безумовної бінарної оптимізації. На додаток до широко використовуваного формулювання QUBO (квадратична безумовна бінарна оптимізація), він також підтримує задачі вищого порядку (HUBO). Розв'язувач використовує неваріаційний квантовий алгоритм, виконуючи більшу частину обчислень на квантових пристроях.

Далі наведено докладніші відомості про використовуваний алгоритм і короткий посібник із використання функції, а також результати тестування на різних екземплярах задач різних розмірів і складності.

## Опис
Оптимізатор — це готова до використання реалізація передових алгоритмів квантової оптимізації. Він розв'язує задачі оптимізації, запускаючи сильно стиснені квантові схеми на квантовому апаратному забезпеченні. Це стиснення досягається шляхом введення контрдіабатичних членів в основну еволюцію квантової системи з часом. Алгоритм виконує кілька ітерацій запусків на апаратному забезпеченні для отримання кінцевих розв'язків і поєднує їх із постобробкою. Ці кроки безперешкодно інтегровані в робочий процес Оптимізатора та виконуються автоматично.

### Як працює Квантовий Оптимізатор?
У цьому розділі описано основи реалізованого алгоритму bf-DCQO. Введення в алгоритм також можна знайти на [каналі Qiskit YouTube.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

Алгоритм базується на часовій еволюції квантової системи, яка трансформується з часом, де розв'язок задачі закодовано в основному стані квантової системи наприкінці еволюції. Згідно з [адіабатичною теоремою](https://en.wikipedia.org/wiki/Adiabatic_theorem), ця еволюція має бути повільною, щоб гарантувати перебування системи в основному стані. Дигіталізація цієї еволюції є основою дигіталізованих квантових адіабатичних обчислень (DQA) та сумнозвісного алгоритму QAOA. Однак необхідна повільна еволюція непрактична для зростаючих розмірів задач, оскільки призводить до збільшення глибини схеми. Використовуючи контрдіабатичні протоколи, можна придушити небажані збудження, що виникають під час коротких часів еволюції, залишаючись при цьому в основному стані. Тут дигіталізація цього коротшого часу еволюції призводить до квантових схем меншої глибини з меншою кількістю заплутуючих вентилів.

Схеми алгоритмів bf-DCQO зазвичай використовують до десяти разів менше заплутуючих вентилів, ніж DQA, і в три-чотири рази менше заплутуючих вентилів, ніж стандартні реалізації QAOA. Завдяки меншій кількості вентилів під час виконання схеми на апаратному забезпеченні виникає менше помилок. Тому оптимізатор не потребує використання таких методів, як придушення або пом'якшення помилок. Реалізація цих методів у майбутніх версіях може ще більше підвищити якість розв'язку.

Хоча алгоритм bf-DCQO використовує ітерації, він є неваріаційним. Після кожної ітерації алгоритму вимірюється розподіл станів. Отриманий розподіл використовується для обчислення так званого поля зміщення (bias-field). Поле зміщення дозволяє починати наступну ітерацію зі стану енергії поблизу попередньо знайденого розв'язку. Таким чином алгоритм з кожною ітерацією переходить до розв'язків із нижчою енергією. Як правило, приблизно десяти ітерацій достатньо для збіжності до розв'язку, а загальна кількість необхідних ітерацій значно менша, ніж у варіаційних алгоритмів, і становить порядку приблизно 100 ітерацій.

Оптимізатор поєднує алгоритм bf-DCQO з класичною постобробкою. Після вимірювання розподілу станів виконується локальний пошук. Під час локального пошуку біти виміряного розв'язку випадковим чином перевертаються. Після перевертання оцінюється енергія нового бітового рядка. Якщо енергія нижча, бітовий рядок зберігається як новий розв'язок. Локальний пошук масштабується лише лінійно з кількістю кубітів, тому він є обчислювально дешевим. Оскільки постобробка виправляє локальні перевертання бітів, вона компенсує помилки перевертання бітів, які часто є наслідком недосконалості апаратного забезпечення та помилок зчитування.

### Робочий процес
Нижче наведено схему робочого процесу Квантового Оптимізатора.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Workflow of the Quantum Optimizer")

Використовуючи Квантовий Оптимізатор, розв'язання задачі оптимізації на квантовому апаратному забезпеченні можна звести до:
* Формулювання цільової функції задачі
* Отримання доступу до Оптимізатора через Qiskit Functions
* Запуску Оптимізатора та збору результатів

## Тести продуктивності
Наведені нижче показники тестування демонструють, що Оптимізатор ефективно вирішує задачі з кількістю кубітів до 156, і дають загальний огляд точності та масштабованості оптимізатора для різних типів задач. Зверни увагу, що фактичні показники продуктивності можуть відрізнятися залежно від конкретних характеристик задачі, таких як кількість змінних, щільність і локальність членів цільової функції та порядок полінома.

Наступна таблиця включає коефіцієнт апроксимації (AR) — метрику, що визначається таким чином:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
де $C$ — цільова функція, $C_{\textrm{min}}$, $C_{\textrm{max}}$ — її мінімальне та максимальне значення, а $C^{*}$ — вартість найкращого знайденого розв'язку відповідно. Таким чином, AR=100\% означає, що було отримано основний стан задачі.

| Приклад           | Кількість кубітів | Коефіцієнт апроксимації | Загальний час (с) | Використання середовища виконання (с) | Загальна кількість вимірювань | Кількість ітерацій |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Екземпляри MaxCut із 28, 30 та 32 кубітами запускалися на ibm_sherbrooke. Екземпляри з 80, 100 та 120 кубітами запускалися на процесорі Heron r2.
- Екземпляри HUBO також запускалися на процесорі Heron r2.

Усі еталонні екземпляри доступні на GitHub (див. [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Приклад запуску цих екземплярів можна знайти в [Приклад 3: Еталонні екземпляри](#example-3-benchmark-instances).

## Входи та виходи
### Вхідні дані
Дивись у наступній таблиці всі вхідні параметри, які приймає Квантовий Оптимізатор. Наступний розділ **Параметри** детальніше описує доступні `options`.

| Назва        | Тип                | Опис                                                                                                                                                                                   | Обов'язковий | За замовчуванням                                                                              | Приклад                                                                                                              |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Коефіцієнти задачі оптимізації, сформульованої у форматі QUBO/HUBO або спіновому форматі. Для отримання додаткової інформації про специфікацію задачі дивись [Прийняті формати задач](#accepted-problem-formats)                                       | Так      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Вкажи, чи коефіцієнти задачі подано у бінарному форматі (QUBO/HUBO) чи у спіновому форматі. Два можливих значення: `"spin"` або `"binary"`                                                   | Так      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Назва Backend для виконання запиту                                                                                                                                                  | Так       | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Параметри для керування надсиланням на апаратне забезпечення, наприклад кількість вимірювань. Для отримання додаткових відомостей про конфігурацію параметрів дивись розділ [Параметри](#options) | Ні       | Щоб побачити значення за замовчуванням конфігурації параметрів, дивись розділ [Параметри](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |

#### Прийняті формати задач
Аргументи `problem` та `problem_type` кодують задачу оптимізації виду

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
де

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Вибравши `problem_type = "binary"`, ти вказуєш, що функція вартості подана у `binary` форматі, тобто $D = {0,  1}^{n}$, як у формулюванні QUBO/HUBO.
- З іншого боку, вибравши `problem_type = "spin"`, функція вартості записується у формулюванні Ізінга, де $D = {-1, 1}^{n}$.

Коефіцієнти задачі слід кодувати у словнику таким чином:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Зверни увагу, що ключі словника мають бути рядками, які містять коректний кортеж невід'ємних цілих чисел без повторень.

#### Параметри
Iskay надає можливості тонкого налаштування через необов'язкові параметри. Хоча значення за замовчуванням добре підходять для більшості задач, ти можеш налаштувати поведінку для конкретних вимог:

| Параметр | Тип | За замовчуванням | Опис |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Квантові вимірювання за ітерацію (більше = точніше) |
| **num_iterations** | `int` | 10 | Ітерації алгоритму (більше ітерацій може покращити якість розв'язку) |
| **use_session** | `bool` | True | Використовувати сесії IBM для скорочення часу очікування в черзі |
| **seed_transpiler** | `int` | None | Встановити для відтворюваної компіляції квантової схеми |
| **direct_qubit_mapping** | `bool` | False | Відображати віртуальні кубіти безпосередньо на фізичні кубіти |
| **job_tags** | `List[str]` | None | Власні теги для відстеження завдань |
| **preprocessing_level** | `int` | 0 | Інтенсивність попередньої обробки задачі (0-3) — дивись деталі нижче |
| **postprocessing_level** | `int` | 2 | Рівень уточнення розв'язку (0-2) — дивись деталі нижче |
| **transpilation_level** | `int` | 0 | Кількість спроб оптимізації Transpiler (0-5) — дивись деталі нижче |
| **transpile_only** | `bool` | False | Аналізувати оптимізацію схеми без виконання повного запуску |

**Рівні попередньої обробки (0-3)**: Особливо важливі для більших задач, які наразі не вкладаються в часи когерентності апаратного забезпечення. Вищі рівні попередньої обробки досягають меншої глибини схеми за допомогою апроксимацій при транспіляції задачі:
- **Рівень 0**: Точні, довші схеми
- **Рівень 1**: Добрий баланс між точністю та апроксимацією — відкидаються лише вентилі з кутами в нижньому 10-му перцентилі
- **Рівень 2**: Дещо вища апроксимація — відкидаються вентилі з кутами в нижньому 20-му перцентилі та використовується `approximation_degree=0.95` при транспіляції
- **Рівень 3**: Максимальний рівень апроксимації — відкидаються вентилі в нижньому 30-му перцентилі та використовується `approximation_degree=0.90` при транспіляції

**Рівні транспіляції (0-5)**: Керуй розширеними спробами оптимізації Transpiler для компіляції квантової схеми. Це може призвести до збільшення класичних накладних витрат, і в деяких випадках може не змінити глибину схеми. Значення за замовчуванням `2` загалом призводить до найменшої схеми та є відносно швидким.
- **Рівень 0**: Оптимізація розкладеної схеми DCQO (розташування, маршрутизація, планування)
- **Рівень 1**: Оптимізація `PauliEvolutionGate`, а потім розкладеної схеми DCQO (max_trials=10)
- **Рівень 2**: Оптимізація `PauliEvolutionGate`, а потім розкладеної схеми DCQO (max_trials=15)
- **Рівень 3**: Оптимізація `PauliEvolutionGate`, а потім розкладеної схеми DCQO (max_trials=20)
- **Рівень 4**: Оптимізація `PauliEvolutionGate`, а потім розкладеної схеми DCQO (max_trials=25)
- **Рівень 5**: Оптимізація `PauliEvolutionGate`, а потім розкладеної схеми DCQO (max_trials=50)

**Рівні постобробки (0-2)**: Керуй обсягом класичної оптимізації, яка компенсує помилки перевертання бітів з різною кількістю жадібних проходів локального пошуку:
- **Рівень 0**: 1 прохід
- **Рівень 1**: 2 проходи
- **Рівень 2**: 3 проходи

**Режим тільки транспіляції**: Тепер доступний для користувачів, які хочуть аналізувати оптимізацію схеми без виконання повного запуску квантового алгоритму.

**Приклад власної конфігурації**: Ось як можна налаштувати Iskay з різними параметрами:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Оптимізація seed**: Зверни увагу, що `seed_transpiler` за замовчуванням встановлено в `None`. Це вмикає автоматичний процес оптимізації Transpiler. Якщо значення `None`, система розпочне спробу з кількома seed-значеннями та вибере те, що дає найменшу глибину схеми, використовуючи всю потужність параметра `max_trials` для кожного рівня транспіляції.

**Продуктивність рівнів транспіляції**: Збільшення кількості `max_trials` з вищими значеннями `transpilation_level` неминуче збільшить час транспіляції, але не завжди змінюватиме кінцеву схему — це значною мірою залежить від конкретної структури та складності схеми. Однак для деяких схем/задач різниця між 10 спробами (рівень 1) та 50 спробами (рівень 5) може бути разючою, тому дослідження цих параметрів може бути ключем до успішного знаходження розв'язку.

### Вихідні дані
| Назва  | Тип                | Опис                                                                  | Приклад                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Розв'язок та метадані. Структура залежить від параметра `transpile_only`. |   Дивись "Вміст словника результату" нижче      |

### Вміст словника результату
Структура словника результату залежить від режиму виконання:

| Поле         | Тип                | Режим | Опис                                                                                                             | Приклад |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Стандартний** | Відсортований відображений розв'язок, де ключі — індекси змінних (як рядки), відсортовані чисельно, а значення — відповідні значення змінних (1/-1 для спінових задач, 1/0 для бінарних задач).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Стандартний** | Детальна інформація про розв'язок (дивись деталі нижче)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Стандартний** | Тип задачі оптимізації ('spin' або 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Тільки транспіляція** | Аналіз схеми та деталі транспіляції (дивись деталі нижче) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Стандартне виконання
Коли необов'язковий параметр `transpile_only=False`:

**Словник solution_info:**
- **"bitstring"** (`str`): Необроблене бітове представлення розв'язку.
- **"cost"** (`float`): Значення вартості/енергії, пов'язане з розв'язком.
- **"seed_transpiler"** (`int`): Випадковий seed, використаний для Transpiler, що дав цей результат.
- **"mapping"** (`Dict[int, int]`): Оригінальне відображення кубіт-до-змінної, використане в обчисленнях.
- **"qpu_time"** (`float`, необов'язково): Час виконання на QPU в секундах.

**Примітки щодо відображення змінних:**
- Словник `solution` отримується з бітового рядка розв'язку з використанням об'єкта `mapping` для індексування змінних.
- Коли `problem_type=spin`, використовується присвоєння $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Ключі у словнику розв'язку — це індекси змінних, відсортовані чисельно як рядки.

#### Аналіз транспіляції
Коли необов'язковий параметр `transpile_only=True`:

**Словник transpilation_info:**
- **"best_seed"** (`int`): Оптимальний seed, знайдений для транспіляції
- **"transpilation_time_seconds"** (`float`): Час, витрачений на процес транспіляції
- **"transpiled_circuit"** (`Dict`): Аналіз схеми, що містить:
  - **"depth"** (`int`): Глибина схеми (кількість шарів)
  - **"gate_count"** (`int`): Загальна кількість вентилів у схемі
  - **"num_qubits"** (`int`): Кількість використаних кубітів
  - **"width"** (`int`): Ширина схеми
  - **"operations"** (`Dict[str, int]`): Кількість кожного типу використаного вентиля

**Використання режиму тільки транспіляції:**
- Доступний для користувачів, які хочуть аналізувати оптимізацію схеми без виконання повного запуску квантового алгоритму.
- Корисний для аналізу схем, досліджень оптимізації глибини та розуміння ефектів транспіляції перед тим, як переходити до повного виконання.

## Початок роботи
У цій документації ми пройдемо кроки використання Iskay Quantum Optimizer. У процесі ми швидко покажемо, як завантажити функцію з каталогу і як перетворити задачу на коректний вхід, демонструючи при цьому, як можна експериментувати з різними необов'язковими параметрами.

Для більш детального прикладу ознайомся з навчальним посібником [Розв'язання задачі поділу ринку за допомогою Iskay Quantum Optimizer від Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), де ми проходимо через весь процес використання Iskay Solver для розв'язання задачі поділу ринку, яка представляє реальну задачу розподілу ресурсів, де ринки потрібно розбити на збалансовані регіони продажів для досягнення точних цільових показників попиту.

Автентифікуйся за допомогою свого API-ключа, який можна знайти на [інформаційній панелі IBM Quantum Platform](http://quantum.cloud.ibm.com/), і вибери функцію Qiskit таким чином:

In [ ]:
# ruff: noqa: F821

> **Note:** Наступний код передбачає, що ти зберіг свої облікові дані. Якщо ні, дотримуйся інструкцій у розділі [збережи свій обліковий запис IBM Cloud](/guides/functions#install-qiskit-functions-catalog-client) для автентифікації за допомогою API-ключа.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Приклад 1: Проста функція вартості

Розглянемо функцію вартості у спіновому формулюванні:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

де $(x_0, ..., x_4) \in {-1, 1}^5$.

Розв'язком цієї простої функції вартості є
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
з мінімальним значенням $C^{*} = -6$

### 1. Створення цільової функції

Починаємо зі створення словника з коефіцієнтами цільової функції таким чином:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Запуск Оптимізатора
Розв'язуємо задачу, запускаючи оптимізатор. Оскільки $(x_0, ..., x_4) \in {-1, 1}^5$, необхідно встановити `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Отримання результату
Розв'язок задачі оптимізації надається безпосередньо оптимізатором.

In [ ]:
print(job.result())

Це відобразить словник такого вигляду:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Зверни увагу, що словник `solution` відображає вектор результату $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Приклад 2: MaxCut

Багато задач на графах, таких як MaxCut або максимальна незалежна множина, є NP-важкими задачами та ідеальними кандидатами для тестування квантових алгоритмів і апаратного забезпечення. Цей приклад демонструє розв'язання задачі MaxCut для 3-регулярного графа за допомогою Квантового Оптимізатора.

Для запуску цього прикладу необхідно встановити пакет `networkx` на додаток до `qiskit-ibm-catalog`. Щоб встановити його, виконай наступну команду:

In [ ]:
# %pip install networkx numpy

### 1. Створення цільової функції
Починаємо зі створення випадкового 3-регулярного графа. Для цього графа визначаємо цільову функцію задачі MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Запуск Оптимізатора
Розв'язуємо задачу, запускаючи оптимізатор.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Отримання результату
Отримуємо результат і відображаємо бітовий рядок розв'язку назад на вузли оригінального графа.

In [ ]:
print(job.result())

Розв'язок задачі Maxcut міститься безпосередньо у підсловнику `solution` об'єкта результату

In [ ]:
maxcut_solution = job.result()["solution"]

## Приклад 3: Еталонні екземпляри
Еталонні екземпляри доступні на GitHub: [Еталонні екземпляри Kipu.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Екземпляри можна завантажити за допомогою бібліотеки `pygithub`. Щоб встановити її, виконай таку команду:

In [ ]:
# %pip install pygithub

Шляхи до еталонних екземплярів:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Щоб відтворити продуктивність еталону для екземплярів HUBO, вибери бекенд `ibm_marrakesh` та встанови `direct_qubit_mapping` у значення `True` у підсловнику `options`.
Наступний приклад запускає екземпляр Maxcut із 32 вузлами.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Сценарії використання
Типові сценарії використання для солвера оптимізації — це комбінаторні задачі оптимізації. Ти можеш розв'язувати задачі з багатьох галузей: фінанси, фармацевтика або логістика. Ось кілька прикладів.
* Оптимізація портфелю (QUBO): [наукова публікація](https://doi.org/10.1103/PhysRevApplied.22.054037) та [технічний документ](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Згортання білків (HUBO): [наукова публікація](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Планування логістики (QUBO): [наукова публікація](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Оптимізація мережі: [вебінар](https://www.youtube.com/watch?v=w5SrCIK88No)
* Розподіл ринку (QUBO): [навчальний посібник](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Якщо тебе цікавить розв'язання конкретного сценарію та розробка відповідного маппінгу, ми можемо допомогти. [Зв'яжись з нами.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
## Підтримка
Для отримання підтримки звертайся на support@kipu-quantum.com.
## Наступні кроки
- [Запитати доступ до Quantum Optimizer від Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Спробуй навчальний посібник [Розв'язання задачі розподілу ринку за допомогою Iskay Quantum Optimizer від Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Ознайомся з [Romero, S. V., та ін. (2025). Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Ознайомся з [Cadavid, A. G., та ін. (2024). Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Ознайомся з [Chandarana, P., та ін. (2025). Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)
## Додаткова інформація
Iskay, як і назва нашої компанії Kipu Quantum, — це перуанське слово. Хоча ми є стартапом із Німеччини, ці слова походять із рідної країни одного з наших співзасновників, де Quipu був одним із перших обчислювальних пристроїв, створених людством 2000 років до нашої ери.